# NYC Taxi Trip Duration — Model Comparison

Benchmark different models before Feature Engineering improvements.

In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor

from sklearn.metrics import mean_squared_error, r2_score


In [9]:
df = pd.read_csv("../data/processed_taxi.csv")
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])

df["dayofyear"] = df["pickup_datetime"].dt.dayofyear
categorical_cols = [
    "vendor_id",
    "store_and_fwd_flag",
    "passenger_count",
    "same_location"
]

numerical_cols = [
    "distance_km",
    "lat_diff",
    "lon_diff",
    "hour",
    "dayofweek",
    "month",
    "dayofyear"
]

X = df[categorical_cols + numerical_cols]
y = df["log_trip_duration"]


In [10]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])


In [11]:
models = {
    "Ridge": Ridge(),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        random_state=42
    )
}

In [12]:
results = []

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_val)

    results.append({
        "Model": name,
        "RMSE": np.sqrt(mean_squared_error(y_val, pred)),
        "R2": r2_score(y_val, pred)
    })

pd.DataFrame(results).sort_values("R2", ascending=False)


,Model,RMSE,R2
1,Random Forest,0.514671,0.596386
2,Gradient Boosting,0.517117,0.592541
3,XGBoost,0.525665,0.578959
0,Ridge,0.625679,0.403501


## Next Step
Choose the best model family, then return to Feature Engineering.